In [1]:
# ==========================================
# Global Stock Index Next-Day Prediction
# Full Standalone Script
# ==========================================

import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Loading dataset...")

# -------------------------------
# Step 1: Load dataset
# -------------------------------
df = pd.read_csv("Daily_Global_Stock_Market_Indicators.csv")

# Convert Date column
df['Date'] = pd.to_datetime(df['Date'])

# Sort by index and date
df = df.sort_values(by=['Index_Name', 'Date'])

print("Original dataset shape:", df.shape)

# -------------------------------
# Step 2: Create next-day target
# -------------------------------
df['Next_Close'] = df.groupby('Index_Name')['Close'].shift(-1)

# Remove rows without target
df = df.dropna()

print("After target creation:", df.shape)

# -------------------------------
# Step 3: Feature selection
# -------------------------------
features = [
    'Open',
    'High',
    'Low',
    'Close',
    'Volume',
    'Daily_Change_Percent'
]

X = df[features]
y = df['Next_Close']

# -------------------------------
# Step 4: Time-based split
# -------------------------------
split_index = int(len(X) * 0.8)

X_train = X[:split_index]
X_test = X[split_index:]

y_train = y[:split_index]
y_test = y[split_index:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# -------------------------------
# Step 5: Train model
# -------------------------------
print("\nTraining Random Forest model...")

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Model training complete.")

# -------------------------------
# Step 6: Evaluate model
# -------------------------------
print("\nEvaluating model...")

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nModel Performance")
print("-------------------")
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.4f}")

# -------------------------------
# Step 7: Save model using pickle
# -------------------------------
filename = 'rf_model.pkl'

with open(filename, 'wb') as f:
    pickle.dump(model, f)

print(f"\nModel saved as: {filename}")


Loading dataset...
Original dataset shape: (18270, 9)
After target creation: (18260, 10)
Training samples: 14608
Testing samples: 3652

Training Random Forest model...
Model training complete.

Evaluating model...

Model Performance
-------------------
MAE:  9907.15
RMSE: 11368.70
R2 Score: -0.0041

Model saved as: rf_model.pkl
